In [14]:
# Práctica 1 - Sistemas Inteligentes
# Autor: Aurelio Ortega Tenedor
# Fecha: 14/11/2025
# Opción seleccionada: Age of Empires

# Modulos de python necesarios importados
import random, math, heapq
from IPython.display import HTML, display

##################################### GENERACIÓN Y CARGA DEL TABLERO DE JUEGO ####################################################

# Función generar_tablero(). Genera un tablero aleatorio con parámetros de probabilidad para cada casilla
def generarTablero(tamano=8, numeroSoldados=4, probArbol=0.10, probHoyo=0.05, probEscarpado=0.10, 
                    probFuego=0.08, probAlimento=0.05, probPortal=0.03, probSerpiente=0.05):

    # Crea una matriz cuadrada que incializa toda con "C" que sería cesped
    tablero = [['C' for _ in range(tamano)] for _ in range(tamano)]

    # Recorre toda la celda y genera un número aleatorio entre 0 y 1
    for fila in range(tamano):
        for columna in range(tamano):
            aleatorio = random.random()         
            # Se compara el número aleatorio para determinar que elemento poner en cada casilla 
            # Si ninguna condición se cumpliera, se queda como "C" cesped
            if aleatorio < probArbol: tablero[fila][columna] = 'T'
            elif aleatorio < probArbol+probHoyo: tablero[fila][columna] = 'H'
            elif aleatorio < probArbol+probHoyo+probEscarpado: tablero[fila][columna] = 'E'
            elif aleatorio < probArbol+probHoyo+probEscarpado+probFuego: tablero[fila][columna] = 'F'
            elif aleatorio < probArbol+probHoyo+probEscarpado+probFuego+probAlimento: tablero[fila][columna] = 'A'
            elif aleatorio < probArbol+probHoyo+probEscarpado+probFuego+probAlimento+probPortal: tablero[fila][columna] = 'P'
            elif aleatorio < probArbol+probHoyo+probEscarpado+probFuego+probAlimento+probPortal+probSerpiente: tablero[fila][columna] = 'V'

    # Colocar soldados solo en Cesped para que no reciban daños o mueran al ser colocados
    soldadosColocados = 0
    while soldadosColocados < numeroSoldados:
        fila, columna = random.randint(0, tamano-1), random.randint(0, tamano-1)
        if tablero[fila][columna] == 'C':
            tablero[fila][columna] = 'S'
            soldadosColocados += 1
            
    # Convierte cada fila de lista a string (más facil guardar y leer desde un archivo)
    return [''.join(fila) for fila in tablero]

# Guardar tablero en un archivo a parte
def guardarTablero(tablero, nombre="TableroGenerado.txt"):
    with open(nombre, "w", encoding="utf-8") as f:
        for fila in tablero:
            f.write(fila + "\n")
            # Mensaje de confirmación
    print(f"TABLERO CREADO Y GUARDADO CORRECTAMENTE EN '{nombre}'.") 

# Convertir letras a número
def codigoCasilla(simbolo):
    return {
        'C': 0, 'S': 1, 'T': 2, 'H': 3, 'E': 4, 'F': 5, 'A': 6, 'P': 7, 'V': 8
    }.get(simbolo, 0)                # Si se encontrase algo desconocido, devolvería 0 por defecto (cesped)

# Leer el archivo y convertirlo en una matriz numérica
def convertirMatrizNum(nombre="TableroGenerado.txt"):
    with open(nombre, "r", encoding="utf-8") as archivo:
        lineas = [linea.strip() for linea in archivo.readlines()]  # strip() elimina saltos de línea (\n)
    ancho = len(lineas[0])                                 # Toma el ancho del tablero según la primera fila

    # Convierte cada carácter en su número correspondiente y lo guarda en una matriz (mapa)
    mapa = []
    for linea in lineas:
        filaNumerica = [codigoCasilla(simbolo) for simbolo in linea[:ancho]]
        mapa.append(filaNumerica)
        # Devuelve mapa (matriz númerica) y líneas (matriz de STRINGS original para aplicar A*)
    return mapa, lineas

################################## COSTES DAÑOS Y VISUALIZACIÓN ###########################################

# Costes de movimiento (lo utiliza A* para calcular)
# Como A* solo calcula distancia/coste pero no supervivencia, incluimos costes para cada casilla
costeMovimiento = {
    'C': 1,            # Cesped coste 1, valor normal
    'S': 1,            # Soldado en casilla, pero se puede caminar como casilla normal
    'E': 2,            # Terreno escarpado, A* lo usará solo si no hay alternativa
    'F': 1,            # Fuego es fácil pasar, pero quema y simula daño de vida
    'A': 1,            # Alimento, no penaliza avanzar por su casilla
    'P': 1,            # Portal, no cuesta nada entrar en el 
    'V': 50,           # Serpiente muy penalizada para que A* casi siempre intente no pasar por ella
    'T': math.inf,     # Arbol coste infinito, obstáculo total del mapa
    'H': math.inf      # Hoyo es muerte, prohibido pisar 
}

# Daños en la vida del soldado
dano = {
    'C': 0,            
    'S': 0, 
    'E': 0,
    'F': 1,             # Fuego daño 1 de vida
    'A': -1,            # Alimento cura 1 (elimina 1 daño)
    'P': 0,
    'V': 0,             # Serpiente da veneno, no daño directamente
    'T': 0,
    'H': 999            # Hoyo, muerte instantánea
}

 # Grupo de imagenes utilizadas
imagenesCasillas = {
    0:"./ImagenesCasillas/Cesped.png",
    1:"./ImagenesCasillas/Soldado.png",
    2:"./ImagenesCasillas/Arbol.png",
    3:"./ImagenesCasillas/Hoyo.png",
    4:"./ImagenesCasillas/TerrenoEscarpado.png",
    5:"./ImagenesCasillas/Fuego.png",
    6:"./ImagenesCasillas/Alimento.png",
    7:"./ImagenesCasillas/Portal.png",
    8:"./ImagenesCasillas/SerpienteVenenosa.png"
}

# Función para crear tablero visual

# Sacar el tamaño del tablero
def generarHtmlTablero(matrizNumerica):
    alto, ancho = len(matrizNumerica), len(matrizNumerica[0])
    # Estilo para imágenes
    html = """
    <style>
      img.game { width:37px; height:37px; }
      table { border-collapse:collapse; } td { padding:0; margin:0; }
    </style><table>
    """
    # Recorrer matriz y colocar las imágenes
    for fila in range(alto):
        html += "<tr>"
        for columna in range(ancho):
            imagen = imagenesCasillas.get(matrizNumerica[fila][columna], imagenesCasillas[0])
            html += f'<td><img class="game" src="{imagen}"></td>'
        html += "</tr>"
    return html

# Buscar todas las coordenadas de un tipo para saber dónde está todo
def buscarPosiciones(mapaLetras, letra):
    posiciones = []
    for fila,filaContenido in enumerate(mapaLetras):
        for columna,celda in enumerate(filaContenido):
            if celda == letra:
                posiciones.append((fila,columna))
    return posiciones

############################### APLICACIÓN DEL ALGORITMO A* #########################################

# Definición del algoritmo A*
def algoritmoEstrella(mapaLetras, inicio, meta, listaPortales):
    filas, columnas = len(mapaLetras), len(mapaLetras[0])   # Dimensiones del mapa

    # Coste hasta llegar a cada celda, matriz g guarda el coste del camino a cada celda
    # Todo inicia en infinito porque no hay camino aún excepto celda inicial 0, para evitar caminos peores
    costeAcumulado = [[math.inf]*columnas for _ in range(filas)]
    costeAcumulado[inicio[0]][inicio[1]] = 0

    #Aquí anotamos de dónde venimos par reconstruir ruta final al llegar a la meta
    padre = [[None]*columnas for _ in range(filas)]

    # Heurística. Usamos Manhattan: distancia mínima sin diagonales
    def heuristica(fila, columna):
        return abs(fila-meta[0]) + abs(columna-meta[1])

    # Cola de prioridad guarda costeEstimado y posicion
    colaPrioridad = []
    heapq.heappush(colaPrioridad, (heuristica(inicio[0], inicio[1]), inicio))   # heapq siempre saca la casilla más prometedora

    # Movimientos permitidos solo en cuatro direcciones
    movimientos = [(-1,0),(1,0),(0,-1),(0,1)]
    # Evitar teletransportes rebotando infinitamente
    ultimoPortalSalida = None

    # Bucle principal de A*. Saca la casilla con menor coste estimado
    while colaPrioridad:
        costeEstimado, (x, y) = heapq.heappop(colaPrioridad)

        # Si llegamos a la meta, paramos
        if (x,y) == meta:
            break

        # Portales
        tipoCelda = mapaLetras[x][y]

        # Si estamos en un portal busca los demás portales y quita el último usado para evitar bucle
        if tipoCelda == 'P':
            portalesAlternativos = [p for p in listaPortales if p != (x,y)]
            if ultimoPortalSalida in portalesAlternativos:
                portalesAlternativos.remove(ultimoPortalSalida)

            # Elegimos uno al azar, generando un camino distinto cada vez
            # El coste no aumenta al usar el portal
            if portalesAlternativos:
                nuevoPortal = random.choice(portalesAlternativos)
                
                if costeAcumulado[x][y] < costeAcumulado[nuevoPortal[0]][nuevoPortal[1]]:
                    costeAcumulado[nuevoPortal[0]][nuevoPortal[1]] = costeAcumulado[x][y]
                    padre[nuevoPortal[0]][nuevoPortal[1]] = (x,y)
                    ultimoPortalSalida = nuevoPortal
                    heapq.heappush(colaPrioridad, (costeAcumulado[x][y] + heuristica(nuevoPortal[0], nuevoPortal[1]), nuevoPortal))
                continue

        # Movimientos normales
        for dx,dy in movimientos:
            nuevaFila, nuevaColumna = x+dx, y+dy
            if 0<=nuevaFila<filas and 0<=nuevaColumna<columnas:
                tipoNuevaCelda = mapaLetras[nuevaFila][nuevaColumna]
                costeCelda = costeMovimiento[tipoNuevaCelda]
                # Si el coste del movimiento es infinito no se puede (obstáculo)
                if math.isinf(costeCelda):
                    continue
                # Nuevo coste hasta ese vecino
                nuevoCoste = costeAcumulado[x][y] + costeCelda

                # Si es mejor que el conocido anteriormente, actualizamos, guardamos padre y metemos en cola
                if nuevoCoste < costeAcumulado[nuevaFila][nuevaColumna]:
                    costeAcumulado[nuevaFila][nuevaColumna] = nuevoCoste
                    padre[nuevaFila][nuevaColumna] = (x,y)
                    heapq.heappush(colaPrioridad, (nuevoCoste + heuristica(nuevaFila,nuevaColumna), (nuevaFila,nuevaColumna)))

    # Si no hay un camino posible
    if math.isinf(costeAcumulado[meta[0]][meta[1]]):
        return None, math.inf

    # Reconstrucción del camino. Se retrocede de meta hasta inicio y se invierte la lista
    ruta = []
    nodoActual = meta
    while nodoActual:
        ruta.append(nodoActual)
        nodoActual = padre[nodoActual[0]][nodoActual[1]]
    ruta.reverse()

    # Resultado final, devolviendo ruta óptima y coste final
    return ruta, costeAcumulado[meta[0]][meta[1]]

###################################### BLOQUE PARA CONTROLAR EL JUEGO ##############################################

# ANSI para imprimir el texto en color por consola y dar mejor feedback al usuario
ROJO="\033[91m"
VERDE="\033[92m"
AZUL="\033[94m"
RESET="\033[0m"

# Preguntar el tamaño del tablero al usuario
try:
    tamanoTablero = int(input("Introduce el tamaño del tablero (8,16,32,...): "))
except:
    tamanoTablero = 8    # Si hay un error de lectura, el valor por defecto se establece en 8x8

# Preguntar número de soldados (mínimo 4). Bucle infinito hasta que el usuario introduzca 4 o más
while True:
    try:
        numeroSoldados = int(input("Introduce número de soldados (mínimo 4): "))
        if numeroSoldados < 4:
            print("ERROR: Mínimo 4 soldados.")
            continue
        break
    except:
        print("Introduce número válido.")   # Si escribe otra cosa que no sea número, se reintenta

# Generar el tablero y guardarlo en un .txt
tableroCreado = generarTablero(tamanoTablero, numeroSoldados)
guardarTablero(tableroCreado)

# Convertir en matrices útiles
matrizNumerica, mapaLetras = convertirMatrizNum()
# Detectar las posiciones clave para saber dónde empiezan los soldados y los portales
posicionesSoldados = buscarPosiciones(mapaLetras, 'S')
posicionesPortales = buscarPosiciones(mapaLetras,'P')

# Mostrar información inicial
print(f"\nSoldados detectados: {posicionesSoldados}")
print(f"Portales detectados: {posicionesPortales}")
display(HTML(generarHtmlTablero(matrizNumerica)))

# Seleccionar el camino objetivo. El usuario elige coordenadas dentro del rango del tablero
while True:
    try:
        filaDestino = int(input(f"Fila destino (0..{tamanoTablero-1}): "))
        columnaDestino = int(input(f"Columna destino (0..{tamanoTablero-1}): "))
        
        if not(0 <= filaDestino < tamanoTablero and 0 <= columnaDestino < tamanoTablero):
            print("Fuera del tablero.")    # No se puede posicionar fuera del tablero
            continue
            
        if math.isinf(costeMovimiento[mapaLetras[filaDestino][columnaDestino]]):
            print("Casilla bloqueada.")    # No se permite elegir un árbol o un hoyo como destino
            continue
            
        destinoFinal = (filaDestino,columnaDestino)
        break
    except:
        print("Introduce enteros válidos.")  # Evitar fallos si el usuario introduce otra cosa

############################################ MOVIMIENTO FINAL #################################################

# Cada soldado empieza con 3 de vida y llegados cuenta los soldados que alcanzan el destino
vidaMaxima = 3
llegados = 0

# Bucle para procesar cada soldado uno por uno
for soldado in posicionesSoldados:

    # Decoración visual para el seguimiento por la consola del soldado
    print("\n" + "="*60)
    print(f"PROCESANDO SOLDADO {soldado}")
    print("="*60)

    # Obtener ruta con el algoritmo A*
    ruta, coste = algoritmoEstrella(mapaLetras, soldado, destinoFinal, posicionesPortales)
    if ruta is None:
        print(f"{ROJO}NO HAY RUTA{RESET}")    # Si es imposible se informa y pasa al siguiente soldado
        continue

    # Simulación del daño de la ruta original antes de mover
    vidaSimulada = vidaMaxima
    veneno = False

    # Recorremos la ruta prevista suponiendo el daño real para ver si llegaría vivo
    for (filaRuta, columnaRuta) in ruta[1:]:  # omitir la casilla inicial
        celda = mapaLetras[filaRuta][columnaRuta]

        # Daño directo
        vidaSimulada -= dano[celda]

        # Si pisa serpiente se activa estado envenenado
        if celda == 'V':
            veneno = True

        # Daño por veneno si está activo
        if veneno:
            vidaSimulada -= 1

        # Si ya está muerto no tiene sentido seguir comprobando veneno
        if vidaSimulada <= 0:
            break

    # Localizamos todas las comidas del mapa
    posicionesComidas = buscarPosiciones(mapaLetras,'A')

    # Saber si hay serpiente en la ruta original
    serpienteRuta = any(mapaLetras[fila][columna] == 'V' for (x,y) in ruta)

    # Decidir si necesita comida para salvarse
    necesitaComida = (vidaSimulada <= 0 and not serpienteRuta and len(posicionesComidas) > 0)

    # Buscar comida solo si es necesario y sin serpientes en ruta
    if necesitaComida:
        print(f"soldado {soldado} NO llegaría vivo. Se buscará comida.")

        # Buscar la comida más accesible
        mejorComida, mejorRuta, menorCoste = None, None, 9999
        for comida in posicionesComidas:
            rutaComida, costeComida = algoritmoEstrella(mapaLetras, soldado, comida, posicionesPortales)
            if rutaComida and costeComida < menorCoste:
                mejorComida = comida
                mejorRuta = rutaComida
                menorCoste = costeComida

        # Si encuentra comida accesible se une ruta a comida + ruta a meta
        if mejorComida:
            print(f"Comida accesible en {mejorComida}. Se desviará.")

            rutaMeta, costeMeta = algoritmoEstrella(mapaLetras, mejorComida, destinoFinal, posicionesPortales)
            
            if rutaMeta:
                ruta = mejorRuta + rutaMeta[1:]     # Evitar repetir casilla comida
                coste = menorCoste + costeMeta

    # Serpiente encontrada, se informa pero no se busca comida
    elif serpienteRuta:
        print(" Esta ruta contiene serpiente.")


    # Ejecución real de movimiento con daño real (incluso por veneno)
    vidaActual = vidaMaxima
    estadoEnvenenado = False
    ultimoPortalUsado = None
    muerte = False
    rutaEjecutada = []

    # Recorrer cada paso de la ruta final
    for (filaActual,columnaActual) in ruta:

        tipoCasilla = mapaLetras[filaActual][columnaActual]
        rutaEjecutada.append((filaActual,columnaActual))

        # Daño base de cada casilla
        vidaActual -= dano[tipoCasilla]
        if vidaActual > vidaMaxima: 
            vidaActual = vidaMaxima

        # Si pisa serpiente se activa veneno
        if tipoCasilla == 'V' and not estadoEnvenenado:
            estadoEnvenenado = True
            print(f"{VERDE}Soldado {soldado} ENVENENADO en {(filaActual,columnaActual)}{RESET}")
            print("Continúa moviéndose estando envenenado...")

        # Daño por veneno en cada paso que avanza
        if estadoEnvenenado:
            vidaActual -= 1
            print(f"Veneno en {(filaActual,columnaActual)} VIDA ={vidaActual}")

        # Muerte por hoyo o por falta de vida
        if tipoCasilla == 'H' or vidaActual <= 0:
            print(f"{ROJO}Soldado {soldado} murió en {(filaActual,columnaActual)}{RESET}")
            print(f"Ruta recorrida hasta muerte: {rutaEjecutada}")
            muerte = True
            break

        # Comer alimento recupera vida (hasta el máximo)
        if tipoCasilla == 'A':
            vidaActual = min(vidaMaxima, vidaActual + 1)
            print(f"{AZUL}Soldado {soldado} comió en {(filaActual,columnaActual)}. VIDA ={vidaActual}{RESET}")

    # Si el soldado ha muerto pasamos al siguiente
    if muerte:
        continue

    # Si sobrevive, se imprime el resultado
    llegados += 1
    print(f"\nSOLDADO {soldado} LLEGO A LA META CON {vidaActual} DE VIDA.")
    print(f"COSTE TOTAL: {coste}, PASOS EMPLEADOS: {len(ruta)-1}")
    print(f"RUTA HASTA META: {rutaEjecutada}")
       
###################################### RESUMEN FINAL Y TABLERO RESULTANTE #########################################

# Mostrar resultado global
print("\nRESULTADO FINAL:")
if llegados == len(posicionesSoldados):
    print("TODO EL GRUPO CONSIGUIÓ LLEGAR A META. ENHORABUENA!")   # Victoria perfecta
elif llegados == 0:
    print("NINGÚN SOLDADO HA CONSEGUIDO SOBREVIVIR A LA TRAVESÍA.")  # Derrota total
else:
    print(f"HAN CONSEGUIDO SUPERAR LA TRAVESÍA: {llegados}/{len(posicionesSoldados)}.")  # Llegada de soldados pero no todos

################################### VISTA DEL TABLERO FINAL #################################################

# Copiamos tablero
tableroFinal = [[codigoCasilla(simbolo) for simbolo in fila] for fila in mapaLetras]

# Eliminar soldados antiguos
for (fila, columna) in posicionesSoldados:
    if tableroFinal[fila][columna] == 1:     
        tableroFinal[fila][columna] = 0       

# Si al menos un soldado llega, se marca la meta
metaFila, metaColumna = destinoFinal

if llegados > 0:
    tableroFinal[metaFila][metaColumna] = 1    # poner soldado en la meta
else:
    tableroFinal[metaFila][metaColumna] = 0    # ningún soldado llegó entonces ponemos cesped en la meta

# Mostrar tablero final
print("\n TABLERO FINAL DESPUÉS DEL VIAJE:")
display(HTML(generarHtmlTablero(tableroFinal)))

Introduce el tamaño del tablero (8,16,32,...):  10
Introduce número de soldados (mínimo 4):  5


TABLERO CREADO Y GUARDADO CORRECTAMENTE EN 'TableroGenerado.txt'.

Soldados detectados: [(0, 7), (2, 3), (7, 8), (8, 7), (9, 0)]
Portales detectados: [(4, 0), (6, 3), (8, 9), (9, 6)]


,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,


Fila destino (0..9):  0
Columna destino (0..9):  1



PROCESANDO SOLDADO (0, 7)

SOLDADO (0, 7) LLEGO A LA META CON 2 DE VIDA.
COSTE TOTAL: 11, PASOS EMPLEADOS: 8
RUTA HASTA META: [(0, 7), (0, 6), (0, 5), (1, 5), (1, 4), (1, 3), (1, 2), (1, 1), (0, 1)]

PROCESANDO SOLDADO (2, 3)

SOLDADO (2, 3) LLEGO A LA META CON 2 DE VIDA.
COSTE TOTAL: 5, PASOS EMPLEADOS: 4
RUTA HASTA META: [(2, 3), (2, 2), (1, 2), (1, 1), (0, 1)]

PROCESANDO SOLDADO (7, 8)

SOLDADO (7, 8) LLEGO A LA META CON 1 DE VIDA.
COSTE TOTAL: 15, PASOS EMPLEADOS: 14
RUTA HASTA META: [(7, 8), (7, 7), (6, 7), (5, 7), (4, 7), (3, 7), (3, 6), (3, 5), (3, 4), (3, 3), (2, 3), (2, 2), (1, 2), (1, 1), (0, 1)]

PROCESANDO SOLDADO (8, 7)

SOLDADO (8, 7) LLEGO A LA META CON 1 DE VIDA.
COSTE TOTAL: 15, PASOS EMPLEADOS: 14
RUTA HASTA META: [(8, 7), (7, 7), (6, 7), (5, 7), (4, 7), (3, 7), (3, 6), (3, 5), (3, 4), (3, 3), (2, 3), (2, 2), (1, 2), (1, 1), (0, 1)]

PROCESANDO SOLDADO (9, 0)

SOLDADO (9, 0) LLEGO A LA META CON 2 DE VIDA.
COSTE TOTAL: 16, PASOS EMPLEADOS: 12
RUTA HASTA META: [(9, 0)

,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
,,,,,,,,,
